# 10: LGBM window/cap search

Rol: buscar mejor ventana temporal y RUL cap solo para LGBMRegressor.

No usa test oficial. La seleccion se hace por: menor C-MAPSS score, luego menor RMSE, luego menor dangerous_error_pct y MAE como metrica secundaria.


In [11]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "CMAPSSData").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results" / "FD001"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
VALIDATION_RANDOM_STATE = 42
CUT_RULS = (20, 50, 80, 110, 140)
BASE_WINDOW_SIZE = 30
BASE_RUL_CAP = 125

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [12]:
from src.preprocessed_FD001 import (
    prepare_fd001_temporal_validation_only,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_model,
    metrics_by_rul_bin,
    prediction_frame,
    regression_metrics,
    set_global_seed,
)
from src.fd001_experiment_utils import (
    add_bin_metric_columns,
    available_boosting_factories,
    fit_predict_model,
    lgbm_factory,
    metric_row_from_predictions,
    selection_sort,
)


In [13]:
WINDOW_SIZES = [20, 30, 50]
RUL_CAPS = [100, 125, 150]
OUTPUT_PATH = RESULTS_DIR / "fd001_lgbm_window_cap_search.csv"


In [14]:
rows = []
for window_size in WINDOW_SIZES:
    print(f"Preparando features temporales window={window_size}")
    base_prepared = prepare_fd001_temporal_validation_only(
        data_dir=DATA_DIR,
        eval_size=EVAL_SIZE,
        random_state=VALIDATION_RANDOM_STATE,
        max_rul=None,
        cut_ruls=CUT_RULS,
        window_size=window_size,
    )
    representation = f"temporal_w{window_size}"
    for rul_cap in RUL_CAPS:
        prepared = dict(base_prepared)
        prepared["y_train"] = base_prepared["y_train_raw"].clip(upper=rul_cap).copy()
        print(f"Entrenando LGBMRegressor window={window_size}, rul_cap={rul_cap}")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pred_df = fit_predict_model(
                prepared,
                lgbm_factory(random_state=VALIDATION_RANDOM_STATE),
                "LGBMRegressor",
                representation,
            )
        row, _ = metric_row_from_predictions(
            pred_df,
            extra={
                "model_name": "LGBMRegressor",
                "representation": representation,
                "window_size": window_size,
                "rul_cap": rul_cap,
                "sample_weight_scheme": "none",
                "target_used_for_training": "RUL_capped",
                "n_features": len(prepared["feature_columns"]),
            },
        )
        rows.append(row)

search_results = selection_sort(pd.DataFrame(rows))
search_results.to_csv(OUTPUT_PATH, index=False)
display(search_results)
print(OUTPUT_PATH)


Preparando features temporales window=20
Entrenando LGBMRegressor window=20, rul_cap=100
Entrenando LGBMRegressor window=20, rul_cap=125
Entrenando LGBMRegressor window=20, rul_cap=150
Preparando features temporales window=30
Entrenando LGBMRegressor window=30, rul_cap=100
Entrenando LGBMRegressor window=30, rul_cap=125
Entrenando LGBMRegressor window=30, rul_cap=150
Preparando features temporales window=50
Entrenando LGBMRegressor window=50, rul_cap=100
Entrenando LGBMRegressor window=50, rul_cap=125
Entrenando LGBMRegressor window=50, rul_cap=150


,representation,model_name,n_eval,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct,mae_rul_0_30,dangerous_error_pct_rul_0_30,mae_rul_30_60,dangerous_error_pct_rul_30_60,mae_rul_60_90,dangerous_error_pct_rul_60_90,mae_rul_90plus,dangerous_error_pct_rul_90plus,window_size,rul_cap,sample_weight_scheme,target_used_for_training,n_features
0,temporal_w50,LGBMRegressor,99,11.297658,14.428233,0.883144,261.005416,2.636418,6.060606,4.269154,0.0,6.543172,0.0,14.934006,30.0,15.475423,0.000000,50,125,none,RUL_capped,119
1,temporal_w50,LGBMRegressor,99,10.763671,13.999632,0.889983,350.185992,3.537232,11.111111,3.757967,0.0,6.930890,0.0,16.946397,40.0,13.151239,7.692308,50,150,none,RUL_capped,119
2,temporal_w30,LGBMRegressor,99,12.809901,16.517134,0.846858,397.315013,4.013283,11.111111,4.325694,0.0,8.545874,15.0,16.484272,40.0,17.463165,0.000000,30,125,none,RUL_capped,119
3,temporal_w50,LGBMRegressor,99,15.406920,20.832760,0.756376,586.618713,5.925442,0.000000,4.382780,0.0,7.217168,0.0,10.833797,0.0,27.605390,0.000000,50,100,none,RUL_capped,119
4,temporal_w30,LGBMRegressor,99,15.985341,21.671301,0.736369,655.111791,6.617291,1.010101,3.777642,0.0,7.978818,5.0,9.788049,0.0,29.529707,0.000000,30,100,none,RUL_capped,119
5,temporal_w30,LGBMRegressor,99,13.039733,17.139780,0.835094,807.569714,8.157270,14.141414,4.581104,0.0,9.839609,15.0,22.640281,35.0,14.095223,10.256410,30,150,none,RUL_capped,119
6,temporal_w20,LGBMRegressor,99,14.855676,20.330321,0.767986,856.807621,8.654622,14.141414,6.544768,0.0,14.323650,30.0,17.725707,40.0,17.918703,0.000000,20,125,none,RUL_capped,119
7,temporal_w20,LGBMRegressor,99,17.626795,23.706208,0.684536,971.032492,9.808409,3.030303,6.391740,0.0,10.412030,15.0,10.567330,0.0,30.708480,0.000000,20,100,none,RUL_capped,119
8,temporal_w20,LGBMRegressor,99,15.559932,21.325065,0.744726,1479.919653,14.948683,17.171717,6.344140,0.0,15.399114,25.0,24.120033,45.0,15.978654,7.692308,20,150,none,RUL_capped,119


C:\Users\lauta\OneDrive\Escritorio\ML\TPF-ML\results\fd001_lgbm_window_cap_search.csv
